# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [44]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Managing Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [54]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [55]:

from openai import OpenAI
from pydantic import BaseModel
import os


#Connecting to the OpenAI API using the OpenAI Python SDK
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

#Defining the class for the response format
class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str  # No longer than one paragraph
    Summary: str  # No longer than 1000 tokens
    Tone: str
    InputTokens: int
    OutputTokens: int

# instructions
INSTRUCTIONS = """Extract author, title, AI professional relevance (1 paragraph), and summary (max 1000 tokens)."""
USER_PROMPT_TEMPLATE = "Document:\n\n{context}"

tone = "Old Victorian English"
context_text = "\n\n".join([doc.page_content for doc in docs])

# API call
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(context=context_text)},
    ],
    instructions=f"Write in {tone} style.",
    text_format=ArticleAnalysis
)

event = response.output_parsed



In [47]:
event

ArticleAnalysis(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="Peter F. Drucker’s treatise, 'Managing Oneself,' serves as a foundational text in the realm of self-management within the modern knowledge economy. As organizations relinquish traditional career management roles, individuals are urged to assume responsibility for their own professional trajectories. This shift places emphasis on self-awareness, understanding one's strengths and weaknesses, and navigating one's contributions—to not only excel in one’s career but also to cultivate an enduring sense of fulfillment during an extensive work life. In this age, knowledge workers are challenged to act as their own CEOs, necessitating profound self-knowledge and active engagement in one’s career growth, which delineates a significant intersection for professionals in fields related to leadership and career development.", Summary="In 'Managing Oneself,' Peter F. Drucker elucidates the necessity for individuals in the

In [48]:
event.Summary

"In 'Managing Oneself,' Peter F. Drucker elucidates the necessity for individuals in the modern workforce to take ownership of their careers, especially in an era characterized by rapid change and personal mobility. Drucker asserts that self-knowledge is paramount to professional success and fulfillment in the knowledge economy, where organizations no longer guide careers as they once did. To thrive, individuals must introspectively analyze their strengths, preferred working styles, and core values in order to carve their own paths. He advocates for a technique known as feedback analysis to better discern one’s competencies and areas for growth. Additionally, it is essential to understand how one performs, the environments in which one flourishes, and the contributions one can make. Values alignment with an organization is crucial for long-term satisfaction and effective performance. As the workforce ages and traditional employment paradigms shift, Drucker encourages knowledge workers 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import SummarizationMetric


#Calculating the summarization metric 
test_case = LLMTestCase(input=context_text, actual_output=event.Summary)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?",
        "Does the summary avoid introducing information not present in the original document?",
        "Are the key points from the original document included in the summary?"
    ]
)
metric.measure(test_case)
print(f"Score: {metric.score}")
print(f"Reason: {metric.reason}\n")



Output()

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: any_value. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
test_case_coherence = LLMTestCase(
    input=context_text,
    actual_output=event.Summary
)
metric_coherence = GEval(
    name="Coherence",
    criteria="Coherence - determine if the summary is logically structured, clear, and easy to understand",
    evaluation_steps=[
        "Assess whether ideas flow logically from one to another",
        "Check if sentences are clearly connected with smooth transitions",
        "Verify that language is unambiguous and appropriate",
        "Evaluate the overall structural clarity",
        "Rate coherence on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4o-mini",
    evaluation_questions=[
        "Are the ideas in the summary presented in a logical, easy-to-follow sequence?",
        "Does each sentence connect clearly to the next without abrupt transitions?",
        "Is the language clear and unambiguous throughout the summary?",
        "Are technical or domain-specific terms used appropriately and consistently?",
        "Does the overall structure enhance understanding of the material?"
    ]
)

metric_coherence.measure(test_case_coherence)
print(f"Score: {metric_coherence.score}")
print(f"Reason: {metric_coherence.reason}\n")

In [ ]:
test_case_tonality = LLMTestCase(
    input=context_text,
    actual_output=result.Summary
)
metric_tonality = GEval(
    name="Tonality",
    criteria=f"Tonality - determine if the summary maintains a consistent {tone} tone",
    evaluation_steps=[
        f"Check if the summary consistently uses {tone} language patterns",
        "Assess whether the tone is appropriate for the content",
        "Verify that formality levels remain consistent",
        "Evaluate if the tone enhances reader engagement",
        "Rate tonality consistency on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4o-mini",
    evaluation_questions=[
        f"Does the summary consistently maintain the {tone} style throughout?",
        "Is the tone appropriate for the professional development context?",
        "Does the language choice reflect the intended Old Victorian English characteristics?",
        "Are formality levels consistent across all sections of the summary?",
        "Does the chosen tone enhance or detract from the clarity of the content?"
    ]
)

metric_tonality.measure(test_case_tonality)
print(f"Score: {metric_tonality.score}")
print(f"Reason: {metric_tonality.reason}\n")

In [ ]:
test_case_safety = LLMTestCase(
    input=context_text,
    actual_output=event.Summary
)

metric_safety = GEval(
    name="Safety",
    criteria="Safety - determine if the summary is appropriate, ethical, and free from harmful content",
    evaluation_steps=[
        "Check for any potentially offensive or harmful language",
        "Verify that claims are properly attributed",
        "Assess appropriate handling of sensitive topics",
        "Evaluate ethical representation of source material",
        "Rate safety on a scale of 0 to 1"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4o-mini",
    evaluation_questions=[
        "Is the summary free from potentially harmful or offensive content?",
        "Does the summary avoid making unfounded claims or presenting opinions as facts?",
        "Are sensitive topics (if any) handled appropriately and professionally?",
        "Does the summary maintain ethical standards in representing the original work?",
        "Is the content appropriate for a professional AI development audience?"
    ]
)

metric_safety.measure(test_case_safety)
print(f"Score: {metric_safety.score}")
print(f"Reason: {metric_safety.reason}\n")

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

result = event

# CORRECTNESS METRIC (Basic version)
correctness_metric = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=custom_model,
)

test_case = LLMTestCase(
    input=context_text,
    actual_output=result.Summary
)

evaluate(test_cases=[test_case], metrics=[correctness_metric])

print(f"Correctness Score: {correctness_metric.score}")
print(f"Correctness Reason: {correctness_metric.reason}\n")



Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
